# 09 — Network Reconnaissance & Scanning

**Author:** Bency (Benjamin Adjei)  
**Course:** M.S. Cybersecurity — Network Security / Penetration Testing  
**Tools:** Python 3, socket, subprocess, ipaddress, concurrent.futures

---

## Objectives
- Understand the phases of network reconnaissance
- Perform host discovery (ping sweep) on a subnet
- Conduct TCP port scanning with banner grabbing
- Enumerate services from open ports
- Understand how Nmap maps to these concepts
- Document findings in a structured report

> ⚠️ **Legal Notice:** Only scan networks and systems you own or have explicit written permission to test. Unauthorized scanning is illegal.

## 1. Setup & Imports

In [ ]:
import socket
import ipaddress
import subprocess
import json
import platform
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

## 2. Reconnaissance Phases

| Phase | Goal | Tools |
|-------|------|-------|
| **Passive Recon** | Gather info without touching the target | WHOIS, DNS, Shodan, Google dorking |
| **Active Recon** | Directly probe the target | Ping sweep, port scan, banner grab |
| **Enumeration** | Extract detailed service/version info | Nmap -sV, Netcat, Metasploit |
| **Vulnerability Scan** | Identify known CVEs | Nessus, OpenVAS, Nikto |

### Nmap Scan Types (Reference)
```bash
nmap -sn 192.168.1.0/24          # Ping sweep (host discovery)
nmap -sS 192.168.1.1              # SYN stealth scan
nmap -sV -p 1-1024 192.168.1.1   # Service version detection
nmap -O 192.168.1.1               # OS fingerprinting
nmap -A 192.168.1.1               # Aggressive: OS + version + scripts
```

## 3. Host Discovery — Ping Sweep

In [ ]:
def ping_host(ip: str, timeout: int = 1) -> dict:
    """Send a single ICMP ping to a host. Returns status and latency info."""
    system = platform.system().lower()
    # Windows uses -n, Linux/Mac uses -c; -w/-W sets timeout
    if system == 'windows':
        cmd = ['ping', '-n', '1', '-w', str(timeout * 1000), ip]
    else:
        cmd = ['ping', '-c', '1', '-W', str(timeout), ip]

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout + 2)
        alive = result.returncode == 0
    except subprocess.TimeoutExpired:
        alive = False

    return {'ip': ip, 'alive': alive}


def ping_sweep(network: str, max_workers: int = 50) -> list:
    """Sweep an entire subnet and return list of live hosts."""
    net   = ipaddress.IPv4Network(network, strict=False)
    hosts = [str(h) for h in net.hosts()]
    live  = []

    print(f'Sweeping {network} — {len(hosts)} hosts ...')
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(ping_host, ip): ip for ip in hosts}
        for future in as_completed(futures):
            result = future.result()
            if result['alive']:
                live.append(result['ip'])
                print(f"  [UP] {result['ip']}")

    return sorted(live)


# Demo — scan only localhost to stay safe
live_hosts = ping_sweep('127.0.0.1/32')
print(f'\nLive hosts found: {live_hosts}')

## 4. TCP Port Scanner with Banner Grabbing

In [ ]:
SERVICE_MAP = {
    21: 'FTP',    22: 'SSH',     23: 'Telnet',  25: 'SMTP',
    53: 'DNS',    80: 'HTTP',   110: 'POP3',   143: 'IMAP',
   443: 'HTTPS', 445: 'SMB',   993: 'IMAPS',  995: 'POP3S',
  1433: 'MSSQL',3306: 'MySQL', 3389: 'RDP',  5432: 'PostgreSQL',
  6379: 'Redis', 8080: 'HTTP-Alt', 8443: 'HTTPS-Alt', 27017: 'MongoDB'
}

RISKY_PORTS = {21, 23, 445, 1433, 3389, 6379, 27017}  # Ports often exploited


def scan_port(host: str, port: int, timeout: float = 0.5) -> dict:
    """Attempt TCP connect to host:port. Grab banner if available."""
    result = {
        'port'   : port,
        'state'  : 'closed',
        'service': SERVICE_MAP.get(port, 'unknown'),
        'banner' : None,
        'risk'   : 'HIGH' if port in RISKY_PORTS else 'LOW'
    }
    try:
        with socket.create_connection((host, port), timeout=timeout) as s:
            result['state'] = 'open'
            # Attempt banner grab
            try:
                s.settimeout(1.0)
                banner = s.recv(1024).decode('utf-8', errors='ignore').strip()
                result['banner'] = banner[:100] if banner else None
            except Exception:
                pass
    except (socket.timeout, ConnectionRefusedError, OSError):
        pass
    return result


def scan_host(host: str, ports: list, max_workers: int = 100) -> dict:
    """Scan multiple ports on a host concurrently."""
    open_ports = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(scan_port, host, p): p for p in ports}
        for future in as_completed(futures):
            r = future.result()
            if r['state'] == 'open':
                open_ports.append(r)

    return {
        'host'      : host,
        'scanned_at': datetime.now().isoformat(),
        'ports_scanned': len(ports),
        'open_ports': sorted(open_ports, key=lambda x: x['port'])
    }


# Scan localhost on common ports
TOP_PORTS = [21, 22, 23, 25, 53, 80, 110, 143, 443, 445,
             993, 995, 1433, 3306, 3389, 5432, 6379, 8080, 8443, 27017]

scan_result = scan_host('127.0.0.1', TOP_PORTS)

print(f"Scan of {scan_result['host']} — {scan_result['ports_scanned']} ports\n")
if scan_result['open_ports']:
    for p in scan_result['open_ports']:
        banner_info = f" | Banner: {p['banner']}" if p['banner'] else ''
        risk_flag   = ' ⚠ RISKY' if p['risk'] == 'HIGH' else ''
        print(f"  OPEN  {p['port']:5} / {p['service']:12}{risk_flag}{banner_info}")
else:
    print('  No open ports found on localhost.')

## 5. DNS Enumeration

In [ ]:
def dns_lookup(hostname: str) -> dict:
    """Resolve a hostname to IP addresses."""
    try:
        addr_info = socket.getaddrinfo(hostname, None)
        ips = list({info[4][0] for info in addr_info})
        return {'hostname': hostname, 'ips': ips, 'status': 'resolved'}
    except socket.gaierror as e:
        return {'hostname': hostname, 'ips': [], 'status': f'failed: {e}'}


def reverse_dns(ip: str) -> str:
    """Perform a reverse DNS lookup on an IP address."""
    try:
        return socket.gethostbyaddr(ip)[0]
    except socket.herror:
        return 'No PTR record'


# DNS lookups on public domains (passive recon demo)
targets = ['github.com', 'google.com', 'cloudflare.com']
print('DNS Enumeration Results:\n')
for target in targets:
    result = dns_lookup(target)
    print(f"  {target:20} -> {result['ips']}")
    for ip in result['ips'][:1]:  # Reverse lookup first IP only
        ptr = reverse_dns(ip)
        print(f"    Reverse DNS ({ip}): {ptr}")

## 6. Subdomain Enumeration (Passive)

Brute-force common subdomains to discover exposed services.

In [ ]:
COMMON_SUBDOMAINS = [
    'www', 'mail', 'ftp', 'vpn', 'remote', 'api',
    'dev', 'staging', 'admin', 'portal', 'git', 'jenkins'
]

def enumerate_subdomains(domain: str, wordlist: list) -> list:
    """Check which subdomains resolve for a given domain."""
    found = []
    for sub in wordlist:
        fqdn   = f'{sub}.{domain}'
        result = dns_lookup(fqdn)
        if result['status'] == 'resolved':
            found.append({'subdomain': fqdn, 'ips': result['ips']})
    return found


# Demo — enumerate subdomains of github.com
print('Subdomain enumeration for github.com:\n')
found_subs = enumerate_subdomains('github.com', COMMON_SUBDOMAINS)
for s in found_subs:
    print(f"  [FOUND] {s['subdomain']:35} -> {s['ips']}")
print(f'\nTotal subdomains resolved: {len(found_subs)}')

## 7. Recon Report

In [ ]:
recon_report = {
    'report_generated': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'scanner_host'    : socket.gethostname(),
    'host_discovery'  : {
        'network'   : '127.0.0.1/32',
        'live_hosts': live_hosts
    },
    'port_scan': scan_result,
    'dns_enum' : [
        dns_lookup(t) for t in targets
    ],
    'subdomains_found': found_subs,
    'risk_summary': {
        'risky_open_ports': [
            p for p in scan_result['open_ports'] if p['risk'] == 'HIGH'
        ]
    }
}

print(json.dumps(recon_report, indent=2))

## 8. Python vs Nmap — Feature Comparison

| Feature | Python Script | Nmap |
|---------|--------------|------|
| Ping sweep | ✅ subprocess ping | ✅ `-sn` |
| TCP connect scan | ✅ socket | ✅ `-sT` |
| SYN stealth scan | ❌ needs raw sockets | ✅ `-sS` |
| Banner grabbing | ✅ socket recv | ✅ `-sV` |
| OS fingerprinting | ❌ complex | ✅ `-O` |
| DNS enumeration | ✅ socket | ✅ built-in |
| Speed (threading) | ✅ ThreadPoolExecutor | ✅ highly optimized |
| Custom automation | ✅ full Python | ❌ limited |
| Scripting engine | ❌ | ✅ NSE scripts |

**Use Python** when you need custom logic, integration with other tools, or automated reporting.  
**Use Nmap** when you need speed, stealth scans, or OS detection.

## 9. Key Takeaways

- Always start recon with **passive techniques** before active scanning
- **Ping sweeps** identify live hosts before wasting time port scanning dead IPs
- **Concurrent scanning** (ThreadPoolExecutor) reduces scan time from minutes to seconds
- **Banner grabbing** reveals software versions — useful for matching against CVE databases
- **Risky ports** (RDP/3389, SMB/445, Redis/6379) exposed on internet are prime attack targets
- Document everything — recon reports are essential for penetration test deliverables

---
*Next: 10 — Vulnerability Assessment & CVE Analysis*